# NTRO Threat Detector - Network Flow Metadata Analysis
This notebook demonstrates passive network flow exploration, inter-arrival time (IAT) analysis, throughput metrics, and anomaly/scan identification on extracted unidirectional flow metadata.

In [ ]:
import pandas as pd
import numpy as np
import os

# Load extracted flow metadata
csv_path = os.path.join("..", "data", "processed", "flows.csv")
if not os.path.exists(csv_path):
    csv_path = os.path.join("data", "processed", "flows.csv")

df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} flow records.")
df.head()

## 1. Flow Overview & Protocol Distribution

In [ ]:
print("Protocol Distribution:")
print(df['protocol'].value_counts())

print("\nSummary Statistics:")
df[['flow_duration_sec', 'bytes_per_sec', 'iat_mean', 'total_bytes', 'packet_count']].describe()

## 2. Threat Indicator: Port Scan / Reconnaissance Detection
Inspect single-packet SYN bursts (`syn_count == 1` & `ack_count == 0` & `is_single_packet == 1`).

In [ ]:
syn_scans = df[(df['syn_count'] == 1) & (df['ack_count'] == 0) & (df['is_single_packet'] == 1)]
print(f"Potential Scan Probes Detected: {len(syn_scans)}")
syn_scans[['src_ip', 'dst_ip', 'dst_port', 'protocol', 'tcp_win_init']]

## 3. High-Rate / Long-Duration Flow Analysis
Identify heavy data transfers vs interactive/micro flows.

In [ ]:
top_heavy_flows = df.sort_values(by='total_bytes', ascending=False).head(5)
top_heavy_flows[['flow_id', 'total_bytes', 'flow_duration_sec', 'bytes_per_sec', 'iat_mean']]